# ResStock models: rebuild with the 2025_Release_1 measures and apply HVAC demand flexibility (upgrade 24)

**Purpose:** re-simulate downloaded ResStock models (baseline `data/resstock_models/`, upgrade 4 `data/resstock_models_up04/`; notebook 1) with `Ithaca_2024.epw` through the same build path, optionally applying ResStock upgrade 24 ("HVAC Demand Flexibility with On-Peak Load Shedding"). Pick the scenario in the configuration cell (`SCENARIO`); the source models are never modified. Run the load-table section of notebook 1 on the output folder afterwards to get the 15-min / hourly / annual tables.

| Scenario key | Source | Offset | Output folder |
|---|---|---|---|
| `baseline` | `resstock_models` | 0 (control) | `resstock_models_rebuilt` |
| `up04` | `resstock_models_up04` | 0 (control) | `resstock_models_up04_rebuilt` |
| `up04_df_2F` | `resstock_models_up04` | 2 F | `resstock_models_up04_df` |

Add more entries to `SCENARIOS` for other DF variants.

## What happens per building
1. Copy `models/in.schedules.csv` (if present) and `models/home.xml` from the source folder, and the weather file.
2. Patch `home.xml` / schedules for the 2025_Release_1 measures (see section 2 for each patch).
3. Run an OpenStudio workflow: `ResStockArgumentsPostHPXML` (shifts the heating/cooling setpoints in the schedules file for the on-peak hours; skipped with offset 0) -> `HPXMLtoOpenStudio` (rebuilds the model) -> `set_run_period` -> `add_outputs` -> EnergyPlus.

| # | Section | Output |
|---|---------|--------|
| 1 | Configuration | paths, scenario, flex settings |
| 2 | Helper functions | - |
| 3 | Run simulations (parallel) | `<output folder>/bldgXXXXXXX/run/eplusout.csv` |

## Upgrade 24 settings
`hvac_flex_peak_offset = 2`, `hvac_flex_pre_peak_duration_hours = 0`, `hvac_flex_pre_peak_offset = 0` (pure load shed: heating setpoint -2 °F and cooling setpoint +2 °F during the on-peak window; NY peak hours 18-22 in summer and 17-21 in winter/intermediate months, shifted one hour later during DST).

## Notes
- Run top to bottom, from `grid_edge_fy25/notebooks/`. Windows only.
- Needs the export of ResStock git tag `2025_Release_1` (path in the configuration cell), with `OS_Version` patched to 3.8.0 and the `setKeepSiteLocationInformation` call guarded, so that OpenStudio 3.8.0 / EnergyPlus 24.1 (the same as the existing runs) can be used.
- Compare scenarios only within the rebuilt runs (baseline, up04 and the DF variants), not against the original runs: the rebuild shifts annual electricity by a few percent, and in a different direction for different systems (tested on `bldg0000510`: baseline -2.5 %, up04 +1.7 %). Keep the original runs for validation against the AMI data.
- Vacant homes (`NumberofResidents = 0`; 69 in up04, 111 in the baseline) have no `in.schedules.csv` in the source models (never part of the NREL download). They are processed without it: the default HPXML schedules apply and the flex measure creates a schedule file with only the heating/cooling setpoint columns. Tested on `bldg0006034`.


## Setup
Assumes the project environment is already created and activated (`python -m pip install -r requirements.txt`). The cell below is the same in every notebook of this project. Run notebooks from `grid_edge_fy25/notebooks/` (all paths are relative to it).


In [15]:
# Common setup (identical in all notebooks of this project)
import ast
import json
import os
import re
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

# Paths (relative to grid_edge_fy25/notebooks/)
current_path = Path(os.getcwd())
data_path = current_path / ".." / "data"
base_files = data_path / "base_files"        # building id lists, measures, weather, workflow
input_files = data_path / "input_files"      # NY metadata, AMI consumption
parquet_folder = data_path / "parquet"       # load tables written for the optimization
results_path = current_path / ".." / "results"
extract_dir = data_path / "resstock_models"  # bldgXXXXXXX folders (models + simulation output)

parquet_folder.mkdir(parents=True, exist_ok=True)


---
## 1. Configuration
Set `SCENARIO` in the next cell. All paths are resolved to absolute paths because the OpenStudio workflows run from each building folder.

- `SRC_DIR`: existing ResStock models (read only). `DST_DIR`: rebuilt runs.
- `RESSTOCK_DIR`: export of the ResStock tag `2025_Release_1` (provides `measures/ResStockArgumentsPostHPXML` and `resources/hpxml-measures`).
- `building_ids` come from the building-ID list of the scenario written by notebook 1 (`building_ids_metadata_2024.csv` for the baseline, `building_ids_metadata_2024_up04.csv` for upgrade 4).


In [16]:
# ========================
# CONFIG
# ========================
# Scenario to run. Each entry: source models, building-ID list, destination folder, hvac_flex_peak_offset (deg F).
# Offset 0 = rebuilt control (no flex). Add entries for more DF variants (also set the pre-peak settings below).
SCENARIOS = {
    "baseline":     dict(src="resstock_models",      ids="building_ids_metadata_2024.csv",      dst="resstock_models_rebuilt",      offset=0),
    "up04":         dict(src="resstock_models_up04", ids="building_ids_metadata_2024_up04.csv", dst="resstock_models_up04_rebuilt", offset=0),
    "up04_df_2F":   dict(src="resstock_models_up04", ids="building_ids_metadata_2024_up04.csv", dst="resstock_models_up04_df",      offset=2),
}
SCENARIO = "baseline"      # <- pick one of the keys above

_s = SCENARIOS[SCENARIO]
SRC_DIR = (data_path / _s["src"]).resolve()                   # existing ResStock models (read only)
DST_DIR = (data_path / _s["dst"]).resolve()                   # rebuilt runs are written here
BUILDING_IDS_CSV = (base_files / _s["ids"]).resolve()
EPW_SRC = (base_files / "weather" / "Ithaca_2024.epw").resolve()
MEASURES_SRC = (base_files / "measures").resolve()            # set_run_period, add_outputs

# Export of git tag 2025_Release_1 (OS_Version patched to 3.8.0)
RESSTOCK_DIR = Path(r"C:\Users\jxiong\Documents\GitHub\resstock_2025R1")
OPENSTUDIO_PATH = "C:/openstudio-3.8.0/bin/openstudio.exe"    # edit if installed elsewhere

# Upgrade 24: HVAC demand flexibility, on-peak load shed (offset taken from the scenario above)
HVAC_FLEX_PEAK_OFFSET = _s["offset"]   # hvac_flex_peak_offset (deg F)
HVAC_FLEX_PRE_PEAK_HOURS = 0       # hvac_flex_pre_peak_duration_hours (0 = pure shed)
HVAC_FLEX_PRE_PEAK_OFFSET = 0      # hvac_flex_pre_peak_offset (deg F)
FLEX_RANDOM_SHIFT_MIN = 0          # flex_random_shift_minutes

MAX_PARALLEL = min(8, os.cpu_count() or 4)   # one OpenStudio/EnergyPlus process each
SKIP_FINISHED = True    # skip buildings in DST_DIR that already have run/finished.job and eplusout.csv (set False to re-run everything)

# ========================
# CHECKS
# ========================
for p in [SRC_DIR, BUILDING_IDS_CSV, EPW_SRC, MEASURES_SRC,
          RESSTOCK_DIR / "measures" / "ResStockArgumentsPostHPXML",
          RESSTOCK_DIR / "resources" / "hpxml-measures", Path(OPENSTUDIO_PATH)]:
    if not Path(p).exists():
        raise FileNotFoundError(f"Missing: {p}")
subprocess.run([OPENSTUDIO_PATH, "openstudio_version"], check=True, capture_output=True, text=True)

building_ids = [int(b) for b in pd.read_csv(BUILDING_IDS_CSV)["bldg_id"]]
DST_DIR.mkdir(parents=True, exist_ok=True)

print(f"Scenario '{SCENARIO}': {len(building_ids)} buildings, {SRC_DIR.name} -> {DST_DIR.name}")
print(f"Flex: peak_offset={HVAC_FLEX_PEAK_OFFSET} F, pre_peak_hours={HVAC_FLEX_PRE_PEAK_HOURS}, "
      f"pre_peak_offset={HVAC_FLEX_PRE_PEAK_OFFSET} F, up to {MAX_PARALLEL} simulations at once")


Scenario 'baseline': 766 buildings, resstock_models -> resstock_models_rebuilt
Flex: peak_offset=0 F, pre_peak_hours=0, pre_peak_offset=0 F, up to 8 simulations at once


---
## 2. Helper functions
The up04 models were built with ResStock 3.2 / HPXML 4.0, the flex measure comes from `2025_Release_1`, so a few inputs are adapted. Source files are never changed; only the copies in `DST_DIR` are.

| Patch | Why |
|---|---|
| Derived weather file `Ithaca_2024_noleap2023.epw` (Feb 29 dropped, year 2023, Sunday start) | the schedules have 35040 rows (365 days); the 2024 EPW is a leap year (8784 h) and HPXMLtoOpenStudio rejects the mismatch. Matches the 2023 run period of the original up04 runs |
| Drop all-zero `Vacancy` / `Power Outage` schedule columns | read as unknown text columns by the newer HPXMLtoOpenStudio, which then crashes |
| Drop `DesignLoads`, `EmissionsScenarios`, default `TemperatureCapacitanceMultiplier` from `home.xml` | old element layout rejected by the newer reader / reference ResStock-only emission files; design loads are recomputed |
| Copy of `add_outputs` with EMS update frequency `ZoneTimestep` | EnergyPlus 24.1 rejects `Timestep` for the EMS output variable |


In [17]:
def make_noleap_epw(src: Path, out_dir: Path) -> Path:
    """8784-row AMY 2024 EPW -> 8760-row 2023-calendar EPW (no Feb 29, year 2023, Sunday start)."""
    out_dir.mkdir(parents=True, exist_ok=True)
    dst = out_dir / (src.stem + "_noleap2023.epw")
    lines = src.read_text(encoding="latin-1").splitlines()
    head, data = lines[:8], lines[8:]
    head = [re.sub(r"^(DATA PERIODS,\d+,\d+,[^,]*,)\w+,", r"\1Sunday,", h) for h in head]
    out = []
    for row in data:
        f = row.split(",", 5)
        if f[1] == "2" and f[2] == "29":
            continue
        f[0] = "2023"
        out.append(",".join(f))
    if len(out) != 8760:
        raise RuntimeError(f"{src.name}: expected 8760 rows after dropping Feb 29, got {len(out)}")
    dst.write_text("\n".join(head + out) + "\n", encoding="latin-1")
    return dst


def make_measures_dir(out_dir: Path) -> Path:
    """Copy base_files/measures to out_dir/_measures, with the add_outputs EMS update frequency set to ZoneTimestep."""
    dst = out_dir / "_measures"
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(MEASURES_SRC, dst, ignore=shutil.ignore_patterns("tests", "docs"))
    mrb = dst / "add_outputs" / "measure.rb"
    mrb.write_text(mrb.read_text(encoding="utf-8").replace("setUpdateFrequency('Timestep')",
                                                           "setUpdateFrequency('ZoneTimestep')"), encoding="utf-8")
    return dst


def copy_schedules(src: Path, dst: Path, bldg: str):
    """Copy in.schedules.csv, dropping the legacy all-zero 'Vacancy' / 'Power Outage' columns."""
    drop_names = {"Vacancy", "Power Outage"}
    rows = [ln.rstrip("\r\n").split(",") for ln in src.read_text(encoding="utf-8").splitlines()]
    drop = [i for i, h in enumerate(rows[0]) if h in drop_names]
    for i in drop:
        if any(float(r[i]) != 0 for r in rows[1:]):
            raise RuntimeError(f"{bldg}: column '{rows[0][i]}' has non-zero values; cannot drop")
    keep = [i for i in range(len(rows[0])) if i not in drop]
    dst.write_text("\n".join(",".join(r[i] for i in keep) for r in rows) + "\n", encoding="utf-8")


EPW_USED = make_noleap_epw(EPW_SRC, DST_DIR / "_weather")
MEASURES_DIR = make_measures_dir(DST_DIR)
print("Weather :", EPW_USED)
print("Measures:", MEASURES_DIR)


Weather : C:\Users\jxiong\Documents\GitHub\grid_edge_ithaca\grid_edge_fy25\data\resstock_models_rebuilt\_weather\Ithaca_2024_noleap2023.epw
Measures: C:\Users\jxiong\Documents\GitHub\grid_edge_ithaca\grid_edge_fy25\data\resstock_models_rebuilt\_measures


In [18]:
def build_osw(bdir: Path, bldg: str) -> dict:
    hpxml = (bdir / "models" / "home.xml").as_posix()
    return {
        "measure_paths": [
            (RESSTOCK_DIR / "measures").as_posix(),
            (RESSTOCK_DIR / "resources" / "hpxml-measures").as_posix(),
            MEASURES_DIR.as_posix(),
        ],
        "run_directory": "./run/",
        "steps": [
            {
                "measure_dir_name": "ResStockArgumentsPostHPXML",
                "arguments": {
                    "hpxml_path": hpxml,
                    "hvac_flex_peak_offset": HVAC_FLEX_PEAK_OFFSET,
                    "hvac_flex_pre_peak_duration_hours": HVAC_FLEX_PRE_PEAK_HOURS,
                    "hvac_flex_pre_peak_offset": HVAC_FLEX_PRE_PEAK_OFFSET,
                    "flex_random_shift_minutes": FLEX_RANDOM_SHIFT_MIN,
                    "ev_flex_enabled": False,
                    "building_id": str(int(bldg.replace("bldg", ""))),
                },
            },
            {
                "measure_dir_name": "HPXMLtoOpenStudio",
                "arguments": {"hpxml_path": hpxml, "output_dir": (bdir / "run").as_posix(), "debug": True},
            },
            {
                "measure_dir_name": "set_run_period",
                "arguments": {"timesteps_per_hour": 4, "begin_date": "2023-01-01", "end_date": "2023-12-31"},
            },
            {"measure_dir_name": "add_outputs", "arguments": {}},
        ],
    }


def prepare(bldg: str) -> Path:
    """Create DST_DIR/bldg with patched home.xml, schedules, weather and the workflow OSW; return the OSW path."""
    src, bdir = SRC_DIR / bldg, DST_DIR / bldg
    (bdir / "models").mkdir(parents=True, exist_ok=True)
    (bdir / "weather").mkdir(exist_ok=True)
    # Vacant homes (NumberofResidents = 0) have no in.schedules.csv; the flex measure then creates a schedule file
    # with only the setpoint columns.
    has_sched = (src / "models" / "in.schedules.csv").exists()
    if has_sched:
        copy_schedules(src / "models" / "in.schedules.csv", bdir / "models" / "in.schedules.csv", bldg)
    shutil.copy2(EPW_USED, bdir / "weather" / EPW_USED.name)

    xml = (src / "models" / "home.xml").read_text(encoding="utf-8")
    xml, n1 = re.subn(r"<EPWFilePath>[^<]*</EPWFilePath>", f"<EPWFilePath>{EPW_USED.name}</EPWFilePath>", xml)
    xml, n2 = re.subn(r"<SchedulesFilePath>[^<]*</SchedulesFilePath>",
                      "<SchedulesFilePath>in.schedules.csv</SchedulesFilePath>", xml)
    if n1 != 1 or n2 != (1 if has_sched else 0):
        raise RuntimeError(f"{bldg}: expected 1 EPWFilePath and {1 if has_sched else 0} SchedulesFilePath, got {n1} and {n2}")
    xml = re.sub(r"[ \t]*<DesignLoads\b.*?</DesignLoads>\s*", "", xml, flags=re.S)
    xml = re.sub(r"[ \t]*<EmissionsScenarios>.*?</EmissionsScenarios>\s*", "", xml, flags=re.S)
    xml = re.sub(r"[ \t]*<TemperatureCapacitanceMultiplier[^>]*>1(\.0+)?</TemperatureCapacitanceMultiplier>\s*", "", xml)
    (bdir / "models" / "home.xml").write_text(xml, encoding="utf-8")

    osw = bdir / "workflow_df.osw"
    osw.write_text(json.dumps(build_osw(bdir, bldg), indent=2), encoding="utf-8")
    return osw


def first_error(run_dir: Path) -> str:
    """First '** Severe **' block of eplusout.err (the real cause), else the last ERROR/FATAL line of run.log."""
    err = run_dir / "eplusout.err"
    if err.exists():
        lines = err.read_text(errors="replace").splitlines()
        for i, l in enumerate(lines):
            if "** Severe" in l or "**  Fatal" in l:
                block = [re.sub(r"^\s*\*\*\s*(~~~|Severe|Fatal)\s*\*\*\s*", "", x).strip() for x in lines[i:i + 4]]
                return " ".join(b for b in block if b)[:250]
    log = run_dir / "run.log"
    if log.exists():
        lines = [l for l in log.read_text(errors="replace").splitlines() if "ERROR" in l or "FATAL" in l]
        if lines:
            return lines[-1][:250]
    return ""


---
## 3. Run OpenStudio simulations
Parallel run (`MAX_PARALLEL` at a time), one status line per building:

| Status | Meaning |
|---|---|
| OK | simulation ran; `run/finished.job` written |
| SKIP | already finished in `DST_DIR` (`SKIP_FINISHED`) |
| MISSING | source `home.xml` not available |
| FAIL | workflow failed or an error occurred while preparing the files; the message is the first Severe error of `run/eplusout.err` (full details in `bldgXXXXXXX/run/run.log` and `openstudio_stdout.log`) |

A building counts as failed if the exit code is non-zero, `run/failed.job` exists or `run/eplusout.csv` is missing. Failures are grouped by cause in the summary.


In [19]:
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed


def run_one(bid: int):
    bldg = f"bldg{bid:07d}"
    src, bdir = SRC_DIR / bldg, DST_DIR / bldg
    run_dir = bdir / "run"

    if not (src / "models" / "home.xml").exists():
        return bid, "MISSING", f"no home.xml in {src / 'models'}"
    if SKIP_FINISHED and (run_dir / "finished.job").exists() and (run_dir / "eplusout.csv").exists() \
            and not (run_dir / "failed.job").exists():
        return bid, "SKIP", ""

    try:
        shutil.rmtree(run_dir, ignore_errors=True)   # clear leftovers (e.g. failed.job) from earlier attempts
        osw = prepare(bldg)
        r = subprocess.run([OPENSTUDIO_PATH, "run", "-w", str(osw)], capture_output=True, text=True, cwd=bdir)
        (bdir / "openstudio_stdout.log").write_text(r.stdout + "\n--- stderr ---\n" + r.stderr, encoding="utf-8")
    except Exception as e:
        return bid, "FAIL", f"{type(e).__name__}: {e}"[:250]

    # OpenStudio writes deprecation warnings to stderr, so stderr is not a failure signal.
    ok = r.returncode == 0 and not (run_dir / "failed.job").exists() and (run_dir / "eplusout.csv").exists()
    if ok:
        return bid, "OK", ""
    return bid, "FAIL", first_error(run_dir) or r.stderr.strip()[-200:] or f"rc={r.returncode}"


print(f"Running {len(building_ids)} models from {SRC_DIR.name} -> {DST_DIR.name} (up to {MAX_PARALLEL} at once)...")
results = {"OK": [], "SKIP": [], "MISSING": [], "FAIL": []}

with ThreadPoolExecutor(max_workers=MAX_PARALLEL) as ex:
    futures = [ex.submit(run_one, bid) for bid in building_ids]
    for i, fut in enumerate(as_completed(futures), 1):
        bid, status, msg = fut.result()
        results[status].append((bid, msg))
        print(f"[{i}/{len(building_ids)}] bldg {bid}: {status}" + (f"  {msg}" if status in ("FAIL", "MISSING") else ""))

print({k: len(v) for k, v in results.items()})
if results["FAIL"] or results["MISSING"]:
    by_cause = defaultdict(list)
    for bid, msg in results["FAIL"] + results["MISSING"]:
        by_cause[msg].append(bid)
    print("\nFailed / missing buildings, grouped by first error:")
    for msg, ids in sorted(by_cause.items(), key=lambda kv: -len(kv[1])):
        print(f"  [{len(ids)}] {msg}\n      {sorted(ids)[:15]}{' ...' if len(ids) > 15 else ''}")
else:
    print("All done")


Running 766 models from resstock_models -> resstock_models_rebuilt (up to 8 at once)...
[1/766] bldg 6034: OK
[2/766] bldg 1655: OK
[3/766] bldg 6975: OK
[4/766] bldg 1700: OK
[5/766] bldg 510: OK
[6/766] bldg 7612: OK
[7/766] bldg 6232: OK
[8/766] bldg 3424: OK
[9/766] bldg 13631: OK
[10/766] bldg 7830: OK
[11/766] bldg 11458: OK
[12/766] bldg 9077: OK
[13/766] bldg 12342: OK
[14/766] bldg 12010: OK
[15/766] bldg 10735: OK
[16/766] bldg 11490: OK
[17/766] bldg 15812: OK
[18/766] bldg 15799: OK
[19/766] bldg 17024: OK
[20/766] bldg 15323: OK
[21/766] bldg 13761: OK
[22/766] bldg 19388: OK
[23/766] bldg 17320: OK
[24/766] bldg 19791: OK
[25/766] bldg 22641: OK
[26/766] bldg 21569: OK
[27/766] bldg 23173: OK
[28/766] bldg 25421: OK
[29/766] bldg 21336: OK
[30/766] bldg 23725: OK
[31/766] bldg 26305: OK
[32/766] bldg 26864: OK
[33/766] bldg 31170: OK
[34/766] bldg 31375: OK
[35/766] bldg 29214: OK
[36/766] bldg 34358: OK
[37/766] bldg 32880: OK
[38/766] bldg 34365: OK
[39/766] bldg 31538: